<table>
  <tr>
    <td><div align="left"><font size="30">Camera Calibration using 2D images</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200" height="158" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see Section 13.2.2

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`). If you're running locally you need to have these packages installed.

<div class="alert alert-info">
If you're running on CoLab or in the browser we have to first install machinevisiontoolbox, which is not preinstalled, so the next cell will be slow. In the browser environment it is not possible to examine pixel values by hovering the mouse cursor over an image (this is a limitation of Matplotlib in a Pyodide Web Worker).
</div>

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
from machinevisiontoolbox import *
import numpy as np
np.set_printoptions(linewidth=120, formatter={'float': lambda x: f"{x:8.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"})

The calibration process simply requires a set of images, that contain a chequerboard of known dimensions, shown in various orientations and distance from the camera.

We start by loading a set of camera calibration images distributed with the toolbox, and display them.  The `ImageCollection` is an array of images (not necessarily heterogeneous), `Image.Tile` creates a single image by arranging the multiple images into a grid.

In [ ]:
images = ImageCollection("calibration/*.jpg");
Image.Tile(images).disp()
len(images)

OpenCV provides a function to automatically find the chequerboards.  Given the size of each square, 25mm in this case, we can estimate the camera intrinsic parameters and the lens distortion model parameters

In [ ]:
K, distortion, frames = CentralCamera.images2C(images, gridshape=(7,6), squaresize=25e-3)

The camera intrinsic matrix is

In [ ]:
K

The image centre is given by the top two numbers in the right-hand column

In [ ]:
K[:2, 2]

which we can compare to the coordinates of the centre pixel of the image

In [ ]:
print(images[0].centre)

They are somewhat different, and that tells us that the optical axis of the camera does not pass through the middle pixel of the image.  This is pretty common with most real cameras, so it's never good to assume that's where the principal point is.

We can also plot the estimated pose of the camera, relative to the chequerboard, for each image in the input sequence.

In [ ]:
for frame in frames:
  CentralCamera.plot(pose=frame.pose, scale=0.05)

ax = plt.gca()
ax.set_xlim(-0.2, 0.4)
ax.set_ylim(-0.2, 0.4)
ax.set_zlim(-0.6, 0)

The chequerboard is at the origin (0,0,0) and its z-axis is pointing into the surface of the board.

Now let's look at the estimated distortion data.  The first three numbers are the coefficients of the radial distortion polynomial, while the last two numbers are the coefficients of the tangential distortion polynomial.

In [ ]:
distortion

We will unpack the camera parameters and distortion coefficients into scalar variables to make the following code easier to follow

In [ ]:
u0 = K[0, 2]; v0 = K[1, 2]; fpix_u = K[0, 0]; fpix_v = K[1,1];
k1, k2, p1, p2, k3 = distortion;

The distortion model is explained in more detail in Section 13.6 of **Robotics, Vision & Control: Python**. The strange ordering of the terms in `distortion` is an OpenCV thing.  

We choose to make the resulting undistorted image the same size as the input images.

In [ ]:
U, V = images[12].meshgrid()

the matrices `U` and `V` are coordinate matrices.  They have the same dimensions as the image used to create them, and for any pixel coordinate `(u,v)` the coordinate matrices are such that `U[v,u]=u` and `V[v,u]=v`. **Remember that these are NumPy arrays and the indices are reversed!**

In [ ]:
print(U[100,200], V[100,200])

we convert the image coordinates to retinal image-plane coordinates, actual distances on the sensor chip, in units of metres with respect to the camera's principal point

In [ ]:
u = (U - u0) / fpix_u;
v = (V - v0) / fpix_v;

The radial distance of the pixels from the principal point is simply the Euclidean distance

In [ ]:
r = np.sqrt(u**2 + v**2);

and the retinal coordinate errors due to distortion are given by (13.15) in **Robotics, Vision & Control: Python**

In [ ]:
delta_u = u * (k1*r**2 + k2*r**4 + k3*r**6) + p1*u*v + p2*(r**2 + 2*u**2);
delta_v = v * (k1*r**2 + k2*r**4 + k3*r**6) + p1*(r**2 + 2*v**2) + p2*u*v;

So the undistorted coordinates are

In [ ]:
ud = u + delta_u; vd = v + delta_v;

and we convert those back to image plane coordinates

In [ ]:
Ud = ud * fpix_u + u0;
Vd = vd * fpix_v + v0;

Now `Ud[v,u]` and `Vd[v,u]` are the distorted coordinates of the point `(u,v)` and we can use this to warp the image to correct the distortion

In [ ]:
undistorted = images[12].warp(Ud, Vd)

Image.Hstack((images[12], undistorted)).disp()

In the side-by-side comparison we can see that the edges of, and lines on, the chequerboard have become straighter.

Warping is a complex operation and is described in **Robotics, Vision & Control: Python**.  For `image.warp[U,V]`, `U` and `V` are coordinate matrices as discussed above, and the output image will be the same size.  For every pixel coordinate (u,v) in the output image, its location in the input image `image` is given by (U[v,u],V[v,u]). This is generally a non-integer coordinate so the pixel value is obtained by interpolation from `image`.  Note that it's possible for (U[v,u],V[v,u]) to reference a pixel coordinate that lies outside the bounds of the image -- in this case the interpolated pixel is set to some special value, perhaps NaN or 0.

Another way to visually represent the image distortion is to show how $(\delta_u, \delta_v)$ changes as a function of $(u,v)$

In [ ]:
plt.clf()   # clear 3D plot
plt.quiver(Ud[::50, ::50], Vd[::50, ::50], -delta_u[::50, ::50], -delta_v[::50, ::50]);

which shows a pattern consistent with barrel distortion